# KV Cache & 추론 최적화 - 실습 코드 2: KV Cache 직접 구현 및 메모리 분석

- Tutorial ID: `expand-kv-cache`
- Tutorial: KV Cache & 추론 최적화
- Section ID: `expand-kv-cache-code-2`
- Section: 실습 코드 2: KV Cache 직접 구현 및 메모리 분석


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: KV Cache 직접 구현 및 메모리 분석
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) autoregressive decoding에서 이전 K/V를 재사용해 계산을 줄이는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import math

class KVCacheAttention(nn.Module):
    """KV Cache를 적용한 Attention 구현
    
    Without KV Cache: 각 스텝마다 모든 이전 토큰의 K, V 재계산
    With KV Cache:    이전 K, V를 저장하고 새 토큰만 계산
    """
        def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.scale = math.sqrt(self.d_k)
    
        def forward(self, x, kv_cache=None, use_cache=True):
        """
        x: (batch, seq_len, d_model) — 새로운 토큰만 포함 (cache 사용 시)
        kv_cache: (past_key, past_value) 또는 None
        """
        batch = x.size(0)
        
                Q = self.q_proj(x).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
                K = self.k_proj(x).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
                V = self.v_proj(x).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # KV Cache 결합
        if kv_cache is not None:
            past_K, past_V = kv_cache
                        K = torch.cat([past_K, K], dim=2)  # (batch, heads, past+new, d_k)
                        V = torch.cat([past_V, V], dim=2)
        
        # 새 캐시 저장
        new_kv_cache = (K, V) if use_cache else None
        
        # Attention 계산
                scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        # Causal mask (캐시 사용 시 자동으로 적용됨)
        seq_len_q = Q.size(2)
        seq_len_k = K.size(2)
        if seq_len_q > 1 or kv_cache is None:
                        causal_mask = torch.triu(
                torch.ones(seq_len_q, seq_len_k, device=x.device), diagonal=seq_len_k - seq_len_q + 1
            ).bool()
                        scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), -1e9)
        
        attn = torch.softmax(scores, dim=-1)
                output = torch.matmul(attn, V)
                output = output.transpose(1, 2).contiguous().view(batch, -1, self.n_heads * self.d_k)
        
        return self.out_proj(output), new_kv_cache

# ── 자기회귀 생성 비교 ──
def generate_without_cache(model, input_ids, max_new_tokens=50):
    """KV Cache 없이 생성 (매번 전체 재계산)"""
        tokens = input_ids.clone()
    total_flops = 0
    
        for _ in range(max_new_tokens):
        # 매번 전체 시퀀스 처리
                x = model(tokens)
        next_token = x[:, -1:, :].argmax(dim=-1)  # 단순화
                tokens = torch.cat([tokens, next_token], dim=1)
        total_flops += tokens.size(1)  # 시퀀스 길이에 비례
    
    return tokens, total_flops

def generate_with_cache(model, input_ids, max_new_tokens=50):
    """KV Cache로 생성 (새 토큰만 계산)"""
        tokens = input_ids.clone()
    
    # 첫 번째: 전체 프롬프트 처리
        x = tokens
    attn_out, kv_cache = model(x)
    next_token = attn_out[:, -1:, :].argmax(dim=-1)  # 단순화
        tokens = torch.cat([tokens, next_token], dim=1)
    
    total_flops = input_ids.size(1)  # 초기 처리
    
        for _ in range(max_new_tokens - 1):
        # 이후: 새 토큰만 처리 (O(1) 계산 + O(N) 캐시 읽기)
                x = tokens[:, -1:]  # 마지막 토큰만
        attn_out, kv_cache = model(x, kv_cache=kv_cache)
        next_token = attn_out[:, -1:, :].argmax(dim=-1)
                tokens = torch.cat([tokens, next_token], dim=1)
        total_flops += 1  # 새 토큰 1개만 계산
    
    return tokens, total_flops

# ── 메모리 분석 ──
def analyze_kv_cache_memory(
    n_layers=32, n_heads=32, d_head=128,
    max_seq_len=4096, batch_size=1, dtype_bytes=2
):
    """KV Cache 메모리 사용량 분석"""
    # KV Cache: 2 × n_layers × batch × n_heads × d_head × seq_len × dtype
    bytes_per_token = 2 * n_layers * n_heads * d_head * dtype_bytes
    total_bytes = bytes_per_token * max_seq_len * batch_size
    
    print("=== KV Cache 메모리 분석 ===")
    print(f"모델 설정: {n_layers} layers, {n_heads} heads, d_head={d_head}")
    print(f"시퀀스 길이: {max_seq_len}, 배치: {batch_size}")
    print(f"데이터 타입: FP{dtype_bytes*8}")
    print(f"\n토큰당 KV Cache: {bytes_per_token/1e6:.2f} MB")
    print(f"전체 KV Cache: {total_bytes/1e9:.2f} GB")
    
    # 다양한 시퀀스 길이
    print(f"\n시퀀스 길이별 KV Cache:")
        for seq_len in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
        mem = bytes_per_token * seq_len * batch_size
        print(f"  {seq_len:>6} 토큰: {mem/1e9:.2f} GB")
    
    # FP8 양자화 효과
    print(f"\nFP8 양자화 시:")
        for seq_len in [4096, 8192, 16384]:
        mem_fp16 = bytes_per_token * seq_len * batch_size
        mem_fp8 = mem_fp16 / 2
        print(f"  {seq_len} 토큰: {mem_fp16/1e9:.2f} GB → {mem_fp8/1e9:.2f} GB (50% 절감)")

# 실행
analyze_kv_cache_memory()

# 속도 비교 시뮬레이션
print("\n=== 생성 속도 시뮬레이션 (50 토큰 생성) ===")
seq_len = 100  # 초기 프롬프트 길이
no_cache_flops = sum(seq_len + i for i in range(50))  # 매번 전체 재계산
with_cache_flops = seq_len + 50  # 초기 + 새 토큰 1개씩
print(f"Cache 없이: {no_cache_flops} 단위 연산")
print(f"Cache 있이: {with_cache_flops} 단위 연산")
print(f"속도 향상: {no_cache_flops/with_cache_flops:.1f}x")